# H13 — Kaggle experiment runner

Tests whether the progressive PIC layer-dimension schedule is necessary for small-prefix utility. It compares three fresh MIPIC trainings while holding MRL, SIA, data, optimizer, and evaluation fixed:

1. `progressive`: 16 -> 32 -> 64 -> 256 -> 512 -> 768
2. `constant_small`: 16 at every layer
3. `reversed`: 768 -> 512 -> 256 -> 64 -> 32 -> 16
4. `random_order`: 16 -> 512 -> 32 -> 256 -> 64 -> 768

The notebook runs the complete three-seed paper protocol.

In [1]:
# Kaggle setup; enable a GPU in Notebook options for the full run.
!git clone --depth 1 https://github.com/ManojAnaparthi/MRL_LEXIS.git /kaggle/working/MRL_LEXIS
%cd /kaggle/working/MRL_LEXIS
!pip install -q transformers scikit-learn pandas numpy tqdm

import os, sys, json, gc, random
from pathlib import Path
import torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MIPIC_NUM_WORKERS'] = '0'
os.environ['MIPIC_AMP'] = '1' if torch.cuda.is_available() else '0'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
sys.path.insert(0, '/kaggle/working/MRL_LEXIS')
Path('/kaggle/working/h13_outputs').mkdir(exist_ok=True)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

Cloning into '/kaggle/working/MRL_LEXIS'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 109 (delta 12), reused 104 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 7.66 MiB | 15.70 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/kaggle/working/MRL_LEXIS
device: cuda


In [2]:
SEEDS = [1, 2, 42]
EPOCHS = 5
TRAIN_ROWS = None
PROBE_ROWS = None
MAX_BATCHES = None
BATCH_SIZE = 16
MAX_LENGTH = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'mode': 'paper', 'seeds': SEEDS, 'epochs': EPOCHS, 'device': str(DEVICE)})

{'mode': 'paper', 'seeds': [1, 2, 42], 'epochs': 5, 'device': 'cuda'}


In [3]:
import subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/assignment-3-mipic")
subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/jadhav-anvesh/assignment-3-mipic.git",
     str(REPO_DIR)],
    check=True,
)

Cloning into '/kaggle/working/assignment-3-mipic'...


CompletedProcess(args=['git', 'clone', '--depth', '1', 'https://github.com/jadhav-anvesh/assignment-3-mipic.git', '/kaggle/working/assignment-3-mipic'], returncode=0)

In [4]:
%cd /kaggle/working/assignment-3-mipic
!pip install -q -r requirements-kaggle.txt

/kaggle/working/assignment-3-mipic


In [5]:
from pathlib import Path
import sys

roots = [
    Path("/kaggle/working/assignment-3-mipic"),
    Path("/kaggle/working/MRL_LEXIS"),
    Path.cwd(),
]

input_root = Path("/kaggle/input")
if input_root.exists():
    roots.extend(path.parent for path in input_root.glob("*/evaluate.py"))
    roots.extend(path.parent for path in input_root.glob("*/*/evaluate.py"))

ROOT = next(
    (path.resolve() for path in roots if (path / "evaluate.py").is_file()),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not find evaluate.py. Clone or attach the assignment-3-mipic repository, "
        "then check its Kaggle path."
    )

sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

Repository root: /kaggle/working/assignment-3-mipic


In [6]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, get_scheduler
from evaluate import MRLEvaluationConfig, evaluate_tweeteval
from models.mipic import MIPICConfig, TinyBERT_MIPIC, create_mipic_optimizer
from models.tinybert import TINYBERT_6L_MODEL_NAME
from train import MRLBaselineConfig, build_mrl_train_loader, train_mipic

ROOT = Path('/kaggle/working/MRL_LEXIS')
OUT = Path('/kaggle/working/h13_outputs')
TRAIN_CORPUS = ROOT / 'data/final_data.csv'
PROBE_TRAIN = ROOT / 'data/tweet_train_3k.csv'
PROBE_TEST = ROOT / 'data/tweet_test.csv'
SCHEDULES = {
    'progressive': ((1,16,2,32,3,64,4,256,5,512,6,768),),
    'constant_small': ((1,16,2,16,3,16,4,16,5,16,6,16),),
    'reversed': ((1,768,2,512,3,256,4,64,5,32,6,16),),
    'random_order': ((1,16,2,512,3,32,4,256,5,64,6,768),),
}
tokenizer = AutoTokenizer.from_pretrained(TINYBERT_6L_MODEL_NAME)
torch.set_num_threads(2)

config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

## Run the three schedule experiments

Each arm starts from a fresh seed-controlled model. The probe reports macro-F1 at all nested dimensions. The CKA/SIA and PIC losses are also retained in the training history.

In [7]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def subset_csv(path, rows, seed):
    frame = pd.read_csv(path)
    if rows is not None:
        frame = frame.sample(n=min(rows, len(frame)), random_state=seed).reset_index(drop=True)
    target = OUT / f'{path.stem}_seed{seed}.csv'
    frame.to_csv(target, index=False)
    return target

def run_schedule(name, schedule, seed):
    seed_everything(seed)
    train_path = subset_csv(TRAIN_CORPUS, TRAIN_ROWS, seed)
    cfg = MRLBaselineConfig(batch_size=BATCH_SIZE, epochs=EPOCHS, max_length=MAX_LENGTH, num_workers=0, max_grad_norm=0.0)
    loader = build_mrl_train_loader(train_path, tokenizer, cfg)
    model = TinyBERT_MIPIC(config=MIPICConfig(pic_pipeline_pairs=schedule), device=DEVICE)
    optimizer = create_mipic_optimizer(model)
    scheduler = get_scheduler('cosine_with_min_lr', optimizer=optimizer,
        num_warmup_steps=max(1, int(len(loader)*EPOCHS*0.1)),
        num_training_steps=max(1, len(loader)*EPOCHS),
        scheduler_specific_kwargs={'min_lr': 2e-6})
    previous = os.environ.get('MIPIC_MAX_BATCHES')
    if MAX_BATCHES is None: os.environ.pop('MIPIC_MAX_BATCHES', None)
    else: os.environ['MIPIC_MAX_BATCHES'] = str(MAX_BATCHES)
    try:
        history = train_mipic(model, loader, DEVICE, epochs=EPOCHS, optimizer=optimizer, scheduler=scheduler)
    finally:
        if previous is None: os.environ.pop('MIPIC_MAX_BATCHES', None)
        else: os.environ['MIPIC_MAX_BATCHES'] = previous
    eval_cfg = MRLEvaluationConfig(dimensions=(16,32,64,128,256,512,768), batch_size=64,
        max_length=MAX_LENGTH, train_samples=PROBE_ROWS, test_samples=PROBE_ROWS, max_iter=500)
    probe = evaluate_tweeteval(model.encoder, tokenizer, PROBE_TRAIN, PROBE_TEST, eval_cfg)
    history_rows = [dict(schedule=name, seed=seed, epoch=i+1, **row) for i,row in enumerate(history)]
    probe_rows = [dict(schedule=name, seed=seed, dimension=d, **metrics) for d,metrics in probe.items()]
    del model, optimizer, scheduler, loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return history_rows, probe_rows

all_history, all_probe = [], []
for seed in SEEDS:
    for name, schedule in SCHEDULES.items():
        print(f'===== seed={seed}, schedule={name} =====', flush=True)
        history_rows, probe_rows = run_schedule(name, schedule, seed)
        all_history.extend(history_rows); all_probe.extend(probe_rows)
history_df = pd.DataFrame(all_history)
probe_df = pd.DataFrame(all_probe)
history_df.to_csv(OUT/'h13_training_history.csv', index=False)
probe_df.to_csv(OUT/'h13_probe_results.csv', index=False)
display(probe_df)

===== seed=1, schedule=progressive =====
Training corpus: 20,240 rows; sources={'BANKING77': 3000, 'TWEET_EVAL_SENT': 3000, 'DEEHAN1866/WIC': 3000, 'MRPC': 3000, 'DGEM_FORMAT': 3000, 'STSB': 2893, 'SICK': 2347}
DataLoader: rows=20,240, per_rank_batch=16, world_size=1, effective_global_batch=16, sampler=local


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  287MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  287MB            

[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (32 outputs left out)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_512)
   (layer_2, dim_512) ? (layer_3, dim_32)
   (layer_3, dim_32) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_64)
   (layer_5, dim_64) ? (layer_6, dim_768)
[rank 0] 23:31:49 epoch 1/5: batch 1/1265  |g|=2.578
[rank 0] 23:32:19 epoch 1/5: batch 100/1265  |g|=14.345
[rank 0] 23:32:49 epoch 1/5: batch 200/1265  |g|=7.311
[rank 0] 23:33:20 epoch 1/5: batch 300/1265  |g|=7.356
[rank 0] 23:33:51 epoch 1/5: batch 400/1265  |g|=6.488
[rank 0] 23:34:22 epoch 1/5: batch 500/1265  |g|=6.663
[rank 0] 23:34:53 epoch 1/5: batch 600/1265  |g|=6.483
[rank 0] 23:35:23 epoch 1/5: batch 700/1265  |g|=4.177
[rank 0] 23:35:54 epoch 1/5: batch 800/1265  |g|=2.884
[rank 0] 23:36:25 epoch 1/5: batch 900/1265  |g|=3.864
[rank 0] 23:36:55 epoch 1/5: batch 1000/1265  |g|=17.963
[rank 0] 23:37:26 epoch 1/5: batch 1100/1265  |g|=3.721
[rank 0] 23:37:56 epoch 1/5: batch 1200/1265  |g|=3.569
[rank 0] 23:38:16 epoch 2/

,schedule,seed,dimension,accuracy,f1,probe_n_iter,probe_converged
0,progressive,1,16,0.544580,0.531364,34,True
1,progressive,1,32,0.570513,0.560253,34,True
2,progressive,1,64,0.582168,0.575125,30,True
3,progressive,1,128,0.593240,0.587784,53,True
4,progressive,1,256,0.605478,0.601142,25,True
...,...,...,...,...,...,...,...
79,random_order,42,64,0.612179,0.608190,38,True
80,random_order,42,128,0.616550,0.613890,45,True
81,random_order,42,256,0.633159,0.631611,37,True
82,random_order,42,512,0.644522,0.644862,25,True


In [8]:
small = probe_df[probe_df.dimension.isin([16,32])]
means = small.groupby('schedule').f1.mean().to_dict()
progressive = means.get('progressive', float('nan'))
alternatives = [value for key,value in means.items() if key != 'progressive']
required_alternatives = {'constant_small', 'reversed', 'random_order'}
supported = bool(required_alternatives.issubset(means) and all(progressive > means[name] for name in required_alternatives))
conclusion = {
    'hypothesis': 'The progressive PIC layer-dimension schedule improves small-prefix utility.',
    'verdict': 'supported' if supported else 'not_supported',
    'mode': 'paper', 'seeds': SEEDS, 'epochs': EPOCHS, 'max_batches': MAX_BATCHES,
    'small_prefix_metric': 'mean macro-F1 over 16D and 32D',
    'small_prefix_f1_by_schedule': means,
    'schedule_definitions': {key: [list(stage) for stage in value] for key,value in SCHEDULES.items()},
    'history_file': str(OUT/'h13_training_history.csv'),
    'probe_file': str(OUT/'h13_probe_results.csv'),
}
(OUT/'h13_conclusion.json').write_text(json.dumps(conclusion, indent=2))
print(json.dumps(conclusion, indent=2))

{
  "hypothesis": "The progressive PIC layer-dimension schedule improves small-prefix utility.",
  "verdict": "not_supported",
  "mode": "paper",
  "seeds": [
    1,
    2,
    42
  ],
  "epochs": 5,
  "max_batches": null,
  "small_prefix_metric": "mean macro-F1 over 16D and 32D",
  "small_prefix_f1_by_schedule": {
    "constant_small": 0.5047290966166202,
    "progressive": 0.5639596394651755,
    "random_order": 0.5790248423934904,
    "reversed": 0.5041252285241197
  },
  "schedule_definitions": {
    "progressive": [
      [
        1,
        16,
        2,
        32,
        3,
        64,
        4,
        256,
        5,
        512,
        6,
        768
      ]
    ],
    "constant_small": [
      [
        1,
        16,
        2,
        16,
        3,
        16,
        4,
        16,
        5,
        16,
        6,
        16
      ]
    ],
    "reversed": [
      [
        1,
        768,
        2,
        512,
        3,
        256,
        4,
        64,
     

In [9]:
import pandas as pd

probe_df = pd.read_csv(
    "/kaggle/working/h13_outputs/h13_probe_results.csv"
)

small = probe_df[probe_df["dimension"].isin([16, 32])]
per_seed = small.groupby(["seed", "schedule"])["f1"].mean().unstack()

print("Mean F1 across 16D and 32D, by seed:")
display(per_seed)

for comparison in ["constant_small", "reversed", "random_order"]:
    differences = per_seed["progressive"] - per_seed[comparison]
    print(
        f"progressive - {comparison}: "
        f"mean={differences.mean():.4f}, "
        f"std={differences.std(ddof=1):.4f}, "
        f"seed_differences={differences.to_dict()}"
    )

Mean F1 across 16D and 32D, by seed:


schedule,constant_small,progressive,random_order,reversed
seed,,,,
1,0.489917,0.545809,0.578440,0.497994
2,0.526021,0.558553,0.572742,0.505881
42,0.498250,0.587518,0.585893,0.508501


progressive - constant_small: mean=0.0592, std=0.0285, seed_differences={1: 0.055891736996718344, 2: 0.0325315360863887, 42: 0.0892683554625584}
progressive - reversed: mean=0.0598, std=0.0168, seed_differences={1: 0.047814377399229, 2: 0.05267200574413433, 42: 0.07901684967980405}
progressive - random_order: mean=-0.0151, std=0.0171, seed_differences={1: -0.03263110855871987, 2: -0.01418983810739416, 42: 0.001625337881169031}


## Package outputs

Download the ZIP from the Kaggle output panel after the complete paper run.

In [10]:
!cd /kaggle/working && zip -qr h13_kaggle_results.zip h13_outputs
print('/kaggle/working/h13_kaggle_results.zip')

/kaggle/working/h13_kaggle_results.zip
